# SSR on ZedBoard / PYNQ — single-sample runner

Loads the `ssr` overlay (AXI DMA + `top_ssr` core), streams one audio file
(`on.mem` / `off.mem` / `other.mem`) into the core via DMA, and reads the
classification result over AXI4-Lite (MMIO).

**Files needed in this folder:** `ssr.bit`, `ssr.hwh`, `on.mem`, `off.mem`, `other.mem`.

## 1. Setup — run once
Loads the overlay and defines the helper functions.

In [ ]:
import numpy as np
from pynq import Overlay, allocate

# --- load the overlay (ssr.bit + ssr.hwh must be in this folder) ---
ol = Overlay("ssr.bit")

# --- AXI DMA (MM2S = memory -> stream -> core) ---
dma = ol.axi_dma_0

# --- top_ssr control/result over AXI4-Lite (MMIO) ---
# both Overlay-IP and MMIO expose .read(offset) / .write(offset, value)
try:
    ssr = ol.top_ssr_0
except AttributeError:
    from pynq import MMIO
    ssr = MMIO(0x43C00000, 0x10000)

# register map of top_ssr
CTRL   = 0x00   # bit0 = soft_reset
STATUS = 0x04   # bit0 = output_valid, bit1 = busy, bit2 = s_axis_ready
RESULT = 0x08   # [1:0] = 0:other 1:on 2:off

N_SAMPLES = 16000                  # 1 s @ 16 kHz, one .mem file
files = {"on": "on.mem", "off": "off.mem", "other": "other.mem"}
LABEL = {0: "OTHER", 1: "ON", 2: "OFF"}

def load_mem(path):
    """Read a .mem file: one 4-hex value per line, 16-bit two's complement (Q1.15)."""
    vals = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            v = int(line, 16) & 0xFFFF
            if v & 0x8000:          # sign extension
                v -= 0x10000
            vals.append(v)
    return np.array(vals, dtype=np.int16)

# one reusable DMA buffer (int16, 16-bit stream)
buf = allocate(shape=(N_SAMPLES,), dtype=np.int16)

import time
def classify(name, verbose=True):
    """Stream one file through the core and return the recognised class (0/1/2)."""
    data = load_mem(files[name])
    n = min(len(data), N_SAMPLES)

    ssr.write(CTRL, 1)              # soft reset the core (clears previous result)
    time.sleep(0.001)

    buf[:n] = data[:n]             # fill the DMA buffer
    dma.sendchannel.transfer(buf)  # stream samples -> top_ssr (DMA sets tlast)
    dma.sendchannel.wait()         # wait until all samples are sent

    t0 = time.time()               # wait for the core to finish
    while not (ssr.read(STATUS) & 0x1):
        if time.time() - t0 > 2.0:
            print("  TIMEOUT: no result"); return None

    res = ssr.read(RESULT) & 0x3
    if verbose:
        print(f"{name}.mem  ->  RESULT = {res} ({LABEL[res]})")
    return res

print("Ready. Overlay loaded. Use classify('on' | 'off' | 'other').")

## 2. Run ONE sample
Change `name` to `"on"`, `"off"` or `"other"` and run this cell (Shift+Enter).

In [ ]:
name = "on"

data = load_mem(files[name])
print(f"Samples loaded: {len(data)}")

ssr.write(CTRL, 1)
time.sleep(0.01)  # dłuższy reset

print(f"STATUS after reset: {bin(ssr.read(STATUS))}")

buf[:len(data)] = data
buf[len(data):] = 0

dma.sendchannel.transfer(buf)
dma.sendchannel.wait()
print("DMA transfer done")

print(f"STATUS after DMA: {bin(ssr.read(STATUS))}")

t0 = time.time()
while not (ssr.read(STATUS) & 0x1):
    if time.time() - t0 > 2.0:
        print(f"TIMEOUT — STATUS = {bin(ssr.read(STATUS))}")
        break
else:
    res = ssr.read(RESULT) & 0x3
    print(f"RESULT = {res} ({LABEL[res]})")

## 3. (optional) Run all three

In [ ]:
ssr.write(CTRL, 1)
time.sleep(0.1)

data = load_mem(files["on"])
n = len(data)
buf[:n] = data[:n]
buf[n:] = 0

print(f"samples loaded: {n}")
print(f"buf.dtype: {buf.dtype}")
print(f"buf.nbytes: {buf.nbytes}")
print(f"buf[:5]: {buf[:5]}")

dma.sendchannel.transfer(buf)

t0 = time.time()
while dma.sendchannel.running:
    if time.time() - t0 > 5.0:
        print("DMA timeout - transfer nie skończył!")
        break
    time.sleep(0.001)
print(f"DMA done po {time.time()-t0:.3f}s")
print(f"STATUS po DMA: {bin(ssr.read(STATUS))}")

t0 = time.time()
while not (ssr.read(STATUS) & 0x1):
    if time.time() - t0 > 5.0:
        print(f"TIMEOUT — STATUS = {bin(ssr.read(STATUS))}")
        break
    time.sleep(0.001)
else:
    print(f"RESULT = {ssr.read(RESULT) & 0x3}")